
# Single Newscast Speech Analysis — 30s topic windows

Este notebook analisa apenas o `*_speech.pkl`.

Objetivo desta versão:

- dividir o speech em pseudo-janelas de **30 segundos**;
- atribuir tema a cada janela;
- detetar mudanças temáticas/lexicais entre janelas;
- consolidar blocos finais por tema/discurso.

A parte de **transição pivot → peça/notícia** foi removida de propósito.


In [ ]:

from pathlib import Path
from collections import Counter
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
FEATURES_DIR = DATA_DIR / "features"

# Root folder for speech outputs from multiple newscasts.
SPEECH_OUTPUT_ROOT = BASE_DIR / "outputs_speech_by_video"
SPEECH_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Escolha do telejornal speech
# Altera esta linha se quiseres outro ficheiro.
# ------------------------------------------------------------
TARGET_SPEECH_FILE = "Telejornal_RTP_Jan_13_speech.pkl"

# Create a video id from the speech filename.
# Example:
# Telejornal_RTP_Dec_2_speech.pkl -> Telejornal_RTP_Dec_2
VIDEO_ID = Path(TARGET_SPEECH_FILE).stem.replace("_speech", "")

# Video-specific speech output folder.
OUTPUT_DIR = SPEECH_OUTPUT_ROOT / VIDEO_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TARGET_SPEECH_FILE:", TARGET_SPEECH_FILE)
print("VIDEO_ID:", VIDEO_ID)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


# ------------------------------------------------------------
# Parâmetros principais
# ------------------------------------------------------------
SAMPLE_RATE_FPS = 1  # no dataset, segundo ≈ frame
SPEECH_WINDOW_SECONDS = 30  # pseudo-janelas temporais do speech

# Transições entre janelas de 30s
LEXICAL_CHANGE_QUANTILE = 0.75
SEMANTIC_CHANGE_QUANTILE = 0.75
VOLUME_CHANGE_QUANTILE = 0.75

WINDOW_TRANSITION_SCORE_THRESHOLD = 2

# Consolidação final de blocos speech
MIN_FINAL_BLOCK_SECONDS = 60
MAX_MERGE_GAP_SECONDS = 45
USE_APPROX_EMBEDDINGS = True  # embeddings são aproximados, porque o pickle só tem embedding por segmento original

print("Output dir:", OUTPUT_DIR.resolve())


## 1. Funções de limpeza e dicionários

Limpeza de texto e dicionários para temas, candidatos e partidos.

In [ ]:
STOPWORDS_PT = {
    "de", "a", "o", "que", "e", "do", "da", "em", "um", "para", "com", "não", "nao", "uma", "os", "no", "se", "na",
    "por", "mais", "as", "dos", "como", "mas", "foi", "ao", "ele", "das", "tem", "à", "aos", "seu", "sua",
    "ou", "ser", "quando", "muito", "há", "ha", "nos", "já", "ja", "está", "esta", "estao", "estão",
    "entre", "também", "tambem", "só", "so", "pelo", "pela", "até", "ate", "isso", "este", "esta",
    "num", "numa", "mesmo", "assim", "sobre", "ainda", "foram", "será", "sera", "ter", "têm", "tem",
    "vai", "vão", "vao", "pode", "podem", "porque", "onde", "depois", "todos", "todas"
}

STRUCTURAL_WORDS = {
    "telejornal", "jornal", "noticias", "notícias", "rtp", "sic", "tvi", "cnn", "cm", "cmtv",
    "direto", "directo", "minuto", "minutos", "hora", "horas", "hoje", "amanha", "amanhã",
    "ontem", "agora", "imagem", "imagens", "fonte", "arquivo"
}


def clean_text_pt(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\\n", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_pt(text, remove_stopwords=True, remove_structural=True, min_len=3):
    text = clean_text_pt(text)
    tokens = text.split()
    tokens = [t for t in tokens if len(t) >= min_len]
    tokens = [t for t in tokens if not t.isdigit()]

    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS_PT]

    if remove_structural:
        tokens = [t for t in tokens if t not in STRUCTURAL_WORDS]

    return tokens


def seconds_to_hhmmss(seconds):
    if pd.isna(seconds):
        return None
    seconds = int(round(float(seconds)))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

In [ ]:
# ------------------------------------------------------------
# Candidate / party / theme aliases
# ------------------------------------------------------------

CANDIDATE_ALIASES = {
    "André Ventura": ["andre ventura", "ventura", "lider do chega"],
    "Cotrim Figueiredo": ["cotrim", "cotrim figueiredo", "cotrim de figueiredo", "joao cotrim de figueiredo"],
    "Luís Marques Mendes": ["marques mendes", "luis marques mendes"],
    "Henrique Gouveia e Melo": ["gouveia e melo", "gouveia melo", "henrique gouveia e melo", "almirante gouveia e melo"],
    "António José Seguro": ["antonio jose seguro", "jose seguro", "antonio seguro"],
    "António Filipe": ["antonio filipe"],
    "Catarina Martins": ["catarina martins"],
    "Jorge Pinto": ["jorge pinto"],
}

PARTY_ALIASES = {
    "CHEGA": ["chega", "partido chega"],
    "IL": ["il", "iniciativa liberal", "liberais"],
    "PSD/AD": ["psd", "ad", "alianca democratica", "partido social democrata", "sociais democratas"],
    "PS": ["ps", "partido socialista", "socialistas"],
    "PCP/CDU": ["pcp", "cdu", "partido comunista", "comunistas"],
    "BE": ["be", "bloco de esquerda"],
    "LIVRE": ["livre", "partido livre"],
    "CDS": ["cds", "cds pp", "cds-pp", "centro democratico social"],
}

THEME_ALIASES = {
    "Abertura/Manchetes": [
        "manchetes", "destaques", "sumário", "sumario",
        "em destaque", "abertura", "boa noite"
    ],

    "Eleições/Campanha": [
        "presidenciais", "presidencial",
        "eleições", "eleicoes", "eleição", "eleicao",
        "eleições presidenciais", "eleicoes presidenciais",
        "candidato", "candidata", "candidatos", "candidaturas",
        "campanha", "campanha eleitoral",
        "debate eleitoral", "debates eleitorais",
        "voto", "votos", "eleitores", "urna", "urnas"
    ],

    "Sondagens": [
        "sondagem", "sondagens",
        "barómetro", "barometro",
        "intenção de voto", "intencao de voto",
        "intenções de voto", "intencoes de voto",
        "estimativa eleitoral", "projeção eleitoral", "projecao eleitoral",
        "pontos percentuais"
    ],

    "Governo/Partidos": [
        "governo", "executivo",
        "primeiro ministro", "primeiro-ministro",
        "ministro", "ministra", "ministros",
        "parlamento", "assembleia da república", "assembleia da republica",
        "oposição", "oposicao",
        "líder parlamentar", "lider parlamentar",
        "grupo parlamentar", "maioria absoluta",
        "moção de censura", "mocao de censura",
        "partidos políticos", "partidos politicos"
    ],

    "Saúde": [
        "saúde", "saude",
        "sns", "serviço nacional de saúde", "servico nacional de saude",
        "hospital", "hospitais",
        "médico", "medico", "médicos", "medicos",
        "enfermeiro", "enfermeira", "enfermeiros", "enfermeiras",
        "urgência", "urgencia", "urgências", "urgencias",
        "doente", "doentes", "utente", "utentes",
        "vacina", "vacinas", "covid",
        "lista de espera", "listas de espera"
    ],

    "Economia/Proteção Social": [
        "economia", "económico", "economico",
        "inflação", "inflacao",
        "preços", "precos", "custo de vida",
        "impostos", "irs", "iva", "irc",
        "orçamento", "orcamento", "orçamento do estado", "orcamento do estado",
        "défice", "defice", "dívida pública", "divida publica",
        "salário", "salario", "salários", "salarios",
        "salário mínimo", "salario minimo",
        "pensões", "pensoes", "reformas",
        "segurança social", "seguranca social",
        "subsídio", "subsidio", "subsídios", "subsidios",
        "apoios sociais", "apoio social",
        "empresas", "juros", "taxas de juro",
        "banco de portugal", "bce", "bancos"
    ],

    "Habitação": [
        "habitação", "habitacao",
        "arrendamento", "arrendar",
        "renda", "rendas",
        "senhorio", "senhorios",
        "inquilino", "inquilinos",
        "crédito habitação", "credito habitacao",
        "empréstimo da casa", "emprestimo da casa",
        "preço das casas", "precos das casas",
        "mercado imobiliário", "mercado imobiliario",
        "imobiliário", "imobiliario"
    ],

    "Educação": [
        "educação", "educacao",
        "escola", "escolas",
        "professor", "professores",
        "aluno", "alunos",
        "ensino", "aulas",
        "creche", "creches",
        "universidade", "universidades",
        "estudante", "estudantes",
        "exames nacionais", "ano letivo", "ano lectivo"
    ],

    "Justiça/Segurança": [
        "justiça", "justica",
        "tribunal", "tribunais",
        "polícia", "policia",
        "psp", "gnr", "pj", "polícia judiciária", "policia judiciaria",
        "crime", "crimes", 
        "fraude", "fraudes",
        "sócrates", "socrates",
        "josé sócrates", "jose socrates",
        "homicídio", "homicidio",
        "agressão", "agressao",
        "corrupção", "corrupcao",
        "pgr", "ministério público", "ministerio publico",
        "detido", "detidos", "arguido", "arguidos",
        "prisão", "prisao",
        "caução", "caucao",
        "buscas", "operação policial", "operacao policial",

        # Acidentes e segurança pública
        "acidente", "acidentes",
        "colisão", "colisao",
        "despiste",
        "queda",
        "descarrilamento",
        "vítima", "vitima", "vítimas", "vitimas",
        "ferido", "feridos", "ferida", "feridas",
        "morto", "mortos", "morta", "mortas",
        "falha de segurança", "falha de seguranca",
        "falha técnica", "falha tecnica",
        "investigação", "investigacao",
        "inquérito", "inquerito"
    ],

    "Internacional": [
        "ucrânia", "ucrania",
        "rússia", "russia",
        "guerra na ucrânia", "guerra na ucrania",
        "israel", "gaza", "palestina", "hamas",
        "médio oriente", "medio oriente",
        "trump", "donald trump",
        "casa branca",
        "eua", "estados unidos",
        "bruxelas", "união europeia", "uniao europeia",
        "diplomacia", "diplomacia europeia",
        "diplomático", "diplomatico",
        "diplomática", "diplomatica",
        "relações diplomáticas", "relacoes diplomaticas",
        "nato", "onu",
        "frança", "franca",
        "espanha", "brasil", "china",
        "reino unido", "alemanha",
        "áfrica do sul", "africa do sul",
        "g20", "g7",

        # Política norte-americana
        "republicanos", "republicano",
        "partido republicano",
        "democratas", "democrata",
        "partido democrata",
        "congresso americano",
        "senado americano",
        "câmara dos representantes", "camara dos representantes"
    ],

    "Greves/Trabalho": [
        "greve", "greves",
        "sindicato", "sindicatos",
        "trabalhador", "trabalhadores",
        "protesto", "protestos",
        "manifestação", "manifestacao",
        "manifestantes",
        "contrato coletivo", "contrato colectivo",
        "concertação social", "concertacao social"
    ],

    "Transportes/Mobilidade": [
        "transportes",
        "metro", "metropolitano",
        "comboio", "comboios",
        "cp", "fertagus",
        "autocarro", "autocarros",
        "trânsito", "transito",
        "aeroporto",
        "tap", "tap air portugal",
        "avião", "aviao", "aviões", "avioes",
        "estrada", "autoestrada",
        "portagens",

        # Operadores e transportes urbanos
        "carris",
        "elétrico", "eletrico",
        "ascensor", "ascensores",
        "funicular",

        # Calçada/Elevador da Glória
        "calçada da glória", "calcada da gloria",
        "elevador da glória", "elevador da gloria"
    ],

    "Ambiente/Meteorologia/Proteção Civil": [
        "ambiente",
        "clima", "climático", "climatico",
        "alterações climáticas", "alteracoes climaticas",
        "seca", "chuva", "temporal",
        "inundações", "inundacoes", "cheias",
        "emissões", "emissoes", "poluição", "poluicao",
        "meteorologia", "previsão meteorológica", "previsao meteorologica",
        "temperatura", "vento", "frio", "calor", "neve",
        "incêndio", "incendio", "incêndios", "incendios",
        "bombeiros",
        "proteção civil", "protecao civil",
        "chamas", "evacuação", "evacuacao"
    ],

    "Desporto": [
        "futebol",
        "benfica", "sporting", "fc porto",
        "liga dos campeões", "liga dos campeoes",
        "liga portuguesa",
        "campeonato nacional",
        "seleção nacional", "selecao nacional",
        "treinador", "jogador", "jogadores",
        "golo", "golos",
        "estádio", "estadio"
    ],

    "Cultura": [
        "cultura",
        "cinema", "teatro",
        "música", "musica",
        "festival", "festivais",
        "livro", "livros",
        "exposição", "exposicao",
        "museu", "museus",
        "artista", "artistas",
        "concerto", "concertos"
    ],
}


def normalize_alias(alias):
    return clean_text_pt(alias)


def contains_alias(text, aliases):
    if not isinstance(text, str):
        return False
    clean = clean_text_pt(text)
    for alias in aliases:
        alias_norm = normalize_alias(alias)
        if not alias_norm:
            continue
        pattern = r"(?<!\w)" + re.escape(alias_norm) + r"(?!\w)"
        if re.search(pattern, clean):
            return True
    return False


def count_alias_hits(text, alias_dict):
    clean = clean_text_pt(text)
    counts = {}
    for label, aliases in alias_dict.items():
        total = 0
        for alias in aliases:
            alias_norm = normalize_alias(alias)
            if not alias_norm:
                continue
            pattern = r"(?<!\w)" + re.escape(alias_norm) + r"(?!\w)"
            total += len(re.findall(pattern, clean))
        counts[label] = int(total)
    return counts


def labels_present(text, alias_dict):
    counts = count_alias_hits(text, alias_dict)
    return [label for label, value in counts.items() if value > 0]


def concat_unique_non_empty(values, max_items=10):
    out = []
    for value in values:
        if pd.isna(value) or str(value).strip() == "":
            continue
        for item in str(value).split(","):
            item = item.strip()
            if item and item not in out:
                out.append(item)
            if len(out) >= max_items:
                break
        if len(out) >= max_items:
            break
    return ", ".join(out)


def split_items(value):
    if pd.isna(value) or str(value).strip() == "":
        return set()
    return {x.strip() for x in str(value).split(",") if x.strip()}


def theme_group(theme):
    politics = {"Eleições/Campanha", "Sondagens", "Governo/Partidos"}
    social = {"Saúde", "Educação", "Habitação", "Economia/Proteção Social"}
    risk = {"Justiça/Segurança", "Ambiente/Meteorologia/Proteção Civil"}

    if theme in politics:
        return "politics"
    if theme in social:
        return "social"
    if theme in risk:
        return "risk"
    return theme

print("N candidates:", len(CANDIDATE_ALIASES))
print("N parties:", len(PARTY_ALIASES))
print("N themes:", len(THEME_ALIASES))

In [ ]:
import sys
import pandas, numpy, cv2, numexpr

print(sys.executable)
print("pandas", pandas.__version__)
print("numpy", numpy.__version__)
print("cv2", cv2.__version__)
print("numexpr", numexpr.__version__)

In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn

print(sys.executable)
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scipy", scipy.__version__)
print("sklearn OK")

## 2. Carregar e normalizar o pickle speech

Cada linha do pickle representa um segmento de fala com `timestamp`, `duration`, `transcript` e `text_embedding`.

In [ ]:
speech_files = sorted(FEATURES_DIR.glob("*_speech.pkl"))
print("N speech files found:", len(speech_files))
if speech_files:
    print("Available speech files:")
    for f in speech_files[:20]:
        print("-", f.name)

speech_path = FEATURES_DIR / TARGET_SPEECH_FILE

if not speech_path.exists():
    if len(speech_files) == 0:
        raise FileNotFoundError(f"Não encontrei ficheiros *_speech.pkl em {FEATURES_DIR.resolve()}")
    print("\nWARNING: TARGET_SPEECH_FILE não encontrado:", TARGET_SPEECH_FILE)
    print("Vou usar o primeiro speech disponível. Altera TARGET_SPEECH_FILE se necessário.")
    speech_path = speech_files[0]

print("\nUsing target speech file:", speech_path.name)

raw_speech = pd.read_pickle(speech_path)
print("Raw object type:", type(raw_speech))

if not isinstance(raw_speech, pd.DataFrame):
    raise TypeError("O pickle speech não carregou como DataFrame. Verifica a estrutura do ficheiro.")

speech_raw_df = raw_speech.copy()
print("Raw speech shape:", speech_raw_df.shape)
print("Raw speech columns:", list(speech_raw_df.columns))
display(speech_raw_df.head())

In [ ]:
def infer_speech_columns(df):
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}

    start_candidates = ["timestamp", "start", "start_time", "start_sec", "start_seconds", "time"]
    duration_candidates = ["duration", "dur", "duration_sec", "duration_seconds"]
    end_candidates = ["end", "end_time", "end_sec", "end_seconds"]
    transcript_candidates = ["transcript", "text", "whisper", "sentence", "content"]
    embedding_candidates = ["text_embedding", "transcript_embedding", "embedding", "embeddings", "vector"]

    def pick(candidates):
        for c in candidates:
            if c in lower:
                return lower[c]
        return None

    start_col = pick(start_candidates)
    duration_col = pick(duration_candidates)
    end_col = pick(end_candidates)
    transcript_col = pick(transcript_candidates)
    embedding_col = pick(embedding_candidates)

    if start_col is None:
        raise ValueError("Não consegui inferir a coluna de início temporal.")
    if transcript_col is None:
        raise ValueError("Não consegui inferir a coluna de transcript/texto.")

    return start_col, duration_col, end_col, transcript_col, embedding_col


START_COL, DURATION_COL, END_COL, TRANSCRIPT_COL, EMBEDDING_COL = infer_speech_columns(speech_raw_df)

print("START_COL:", START_COL)
print("DURATION_COL:", DURATION_COL)
print("END_COL:", END_COL)
print("TRANSCRIPT_COL:", TRANSCRIPT_COL)
print("EMBEDDING_COL:", EMBEDDING_COL)

speech = speech_raw_df.copy()

speech["segment_id"] = np.arange(1, len(speech) + 1)
speech["start_sec"] = pd.to_numeric(speech[START_COL], errors="coerce")

if DURATION_COL is not None:
    speech["duration_sec"] = pd.to_numeric(speech[DURATION_COL], errors="coerce")
    speech["end_sec"] = speech["start_sec"] + speech["duration_sec"]
elif END_COL is not None:
    speech["end_sec"] = pd.to_numeric(speech[END_COL], errors="coerce")
    speech["duration_sec"] = speech["end_sec"] - speech["start_sec"]
else:
    # fallback: usa início do segmento seguinte como fim aproximado
    speech["end_sec"] = speech["start_sec"].shift(-1)
    speech.loc[speech.index[-1], "end_sec"] = speech.loc[speech.index[-1], "start_sec"]
    speech["duration_sec"] = speech["end_sec"] - speech["start_sec"]

speech["start_sec"] = speech["start_sec"].round().astype("Int64")
speech["end_sec"] = speech["end_sec"].round().astype("Int64")
speech["duration_sec"] = (speech["end_sec"] - speech["start_sec"]).astype("Int64")

speech["start_time"] = speech["start_sec"].apply(seconds_to_hhmmss)
speech["end_time"] = speech["end_sec"].apply(seconds_to_hhmmss)
speech["duration_min"] = speech["duration_sec"].astype(float) / 60

speech["start_frame"] = speech["start_sec"].astype(int)
speech["end_frame"] = (speech["end_sec"].astype(int) - 1).clip(lower=speech["start_frame"].astype(int))

speech["transcript"] = speech[TRANSCRIPT_COL].astype(str)
speech["clean_transcript"] = speech["transcript"].apply(clean_text_pt)
speech["tokens"] = speech["clean_transcript"].apply(tokenize_pt)
speech["n_words"] = speech["tokens"].apply(len)

speech = speech.sort_values("start_sec").reset_index(drop=True)

print("Normalized speech shape:", speech.shape)
display(speech[["segment_id", "start_time", "end_time", "duration_sec", "start_frame", "end_frame", "n_words", "transcript"]].head(10))


## Nota sobre a granularidade do speech

O pickle speech não tem timestamp por palavra. Tem apenas `timestamp`, `duration`, `transcript` e `text_embedding` por segmento original.

Por isso, quando dividimos em janelas de 30 segundos, o texto de cada segmento é distribuído de forma **aproximada**, assumindo ritmo de fala uniforme dentro do segmento.

Isto é melhor do que usar diretamente segmentos originais muito longos, mas continua a ser uma aproximação.



## 3. Análise inicial dos segmentos originais

Antes de criar as janelas de 30s, vemos os segmentos originais do pickle.


In [ ]:

display(speech[["segment_id", "start_time", "end_time", "duration_sec", "n_words", "transcript"]])

plt.figure(figsize=(12, 4))
plt.hist(speech["duration_sec"].dropna().astype(float), bins=20)
plt.title("Distribuição da duração dos segmentos speech originais")
plt.xlabel("Duração do segmento original (s)")
plt.ylabel("Número de segmentos")
plt.show()

print("N segmentos originais:", len(speech))
print("Duração média dos segmentos originais:", round(float(speech["duration_sec"].mean()), 2), "s")
print("Duração mediana dos segmentos originais:", round(float(speech["duration_sec"].median()), 2), "s")



## 4. Criar pseudo-janelas speech de 30 segundos

Aqui transformamos os segmentos originais em janelas temporais fixas.

Como não há timestamp por palavra, cada palavra é posicionada aproximadamente dentro do segmento original:

`posição estimada = start_sec + proporção_da_palavra_no_segmento * duration_sec`

Depois cada palavra é atribuída à janela de 30s correspondente.


In [ ]:

def split_transcript_words(text):
    """Divide transcript em palavras preservando a forma original para reconstruir texto legível."""
    if pd.isna(text):
        return []
    return re.findall(r"\S+", str(text))


def to_vector(value):
    """Converte embedding para numpy array, quando existir."""
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    try:
        arr = np.asarray(value, dtype=float)
        if arr.ndim != 1 or arr.size == 0:
            return None
        if np.any(pd.isna(arr)):
            return None
        return arr
    except Exception:
        return None


# ------------------------------------------------------------
# 4.1. Expandir transcripts para palavras com tempo aproximado
# ------------------------------------------------------------
word_rows = []

for _, row in speech.iterrows():
    words = split_transcript_words(row["transcript"])

    if len(words) == 0:
        continue

    start = float(row["start_sec"])
    end = float(row["end_sec"])
    duration = max(float(row["duration_sec"]), 1.0)

    for i, word in enumerate(words):
        # ponto médio aproximado da palavra dentro do segmento
        approx_sec = start + ((i + 0.5) / len(words)) * duration
        window_start = int(np.floor(approx_sec / SPEECH_WINDOW_SECONDS) * SPEECH_WINDOW_SECONDS)
        window_end = window_start + SPEECH_WINDOW_SECONDS

        word_rows.append({
            "segment_id": int(row["segment_id"]),
            "word_index_in_segment": i,
            "word": word,
            "approx_sec": approx_sec,
            "window_start_sec": window_start,
            "window_end_sec": window_end,
        })

speech_words = pd.DataFrame(word_rows)

if speech_words.empty:
    raise ValueError("Não foram encontradas palavras nos transcripts.")

print("N palavras distribuídas por janelas:", len(speech_words))
display(speech_words.head(10))


# ------------------------------------------------------------
# 4.2. Criar todas as janelas de 30s no intervalo do speech
# ------------------------------------------------------------
timeline_start = int(np.floor(float(speech["start_sec"].min()) / SPEECH_WINDOW_SECONDS) * SPEECH_WINDOW_SECONDS)
timeline_end = int(np.ceil(float(speech["end_sec"].max()) / SPEECH_WINDOW_SECONDS) * SPEECH_WINDOW_SECONDS)

window_starts = list(range(timeline_start, timeline_end, SPEECH_WINDOW_SECONDS))

segment_embedding_map = {}
if EMBEDDING_COL is not None:
    for _, row in speech.iterrows():
        segment_embedding_map[int(row["segment_id"])] = to_vector(row[EMBEDDING_COL])

window_rows = []
theme_long_rows = []

grouped_words = dict(tuple(speech_words.groupby("window_start_sec")))

for window_id, ws in enumerate(window_starts, start=1):
    we = ws + SPEECH_WINDOW_SECONDS
    group = grouped_words.get(ws, pd.DataFrame(columns=speech_words.columns))

    text = " ".join(group["word"].astype(str).tolist()) if len(group) > 0 else ""
    clean = clean_text_pt(text)
    tokens = tokenize_pt(clean)

    theme_counts = count_alias_hits(clean, THEME_ALIASES)
    candidate_counts = count_alias_hits(clean, CANDIDATE_ALIASES)
    party_counts = count_alias_hits(clean, PARTY_ALIASES)

    theme_hits_total = sum(theme_counts.values())

    if theme_hits_total > 0:
        dominant_theme = max(theme_counts, key=theme_counts.get)
        dominant_theme_score = int(theme_counts[dominant_theme])
    else:
        dominant_theme = "Other/Unknown"
        dominant_theme_score = 0

    themes_present = ", ".join([k for k, v in theme_counts.items() if v > 0])
    candidates_present = ", ".join([k for k, v in candidate_counts.items() if v > 0])
    parties_present = ", ".join([k for k, v in party_counts.items() if v > 0])

    source_segments = sorted(group["segment_id"].dropna().astype(int).unique().tolist()) if len(group) > 0 else []

    # Embedding aproximado da janela: média ponderada pelos segmentos originais que contribuíram palavras.
    # Nota: isto não é um embedding real recalculado para a janela. É apenas aproximação.
    approx_embedding = None
    if USE_APPROX_EMBEDDINGS and EMBEDDING_COL is not None and len(source_segments) > 0:
        vectors = []
        weights = []
        counts_by_segment = group.groupby("segment_id").size().to_dict()
        for sid, weight in counts_by_segment.items():
            vec = segment_embedding_map.get(int(sid))
            if vec is not None:
                vectors.append(vec)
                weights.append(float(weight))
        if len(vectors) > 0:
            try:
                approx_embedding = np.average(np.vstack(vectors), axis=0, weights=np.asarray(weights))
            except Exception:
                approx_embedding = None

    window_rows.append({
        "window_id": window_id,
        "start_sec": ws,
        "end_sec": we,
        "start_time": seconds_to_hhmmss(ws),
        "end_time": seconds_to_hhmmss(we),
        "start_frame": int(ws * SAMPLE_RATE_FPS),
        "end_frame": int((we * SAMPLE_RATE_FPS) - 1),
        "duration_sec": SPEECH_WINDOW_SECONDS,
        "duration_min": SPEECH_WINDOW_SECONDS / 60,
        "n_raw_words": int(len(group)),
        "n_tokens": int(len(tokens)),
        "source_segments": ", ".join(map(str, source_segments)),
        "dominant_theme": dominant_theme,
        "dominant_theme_score": dominant_theme_score,
        "theme_hits_total": int(theme_hits_total),
        "themes_present": themes_present,
        "candidates_present": candidates_present,
        "parties_present": parties_present,
        "text": text,
        "clean_text": clean,
        "tokens": tokens,
        "approx_embedding": approx_embedding,
    })

    for theme, hits in theme_counts.items():
        if hits > 0:
            theme_long_rows.append({
                "window_id": window_id,
                "start_sec": ws,
                "end_sec": we,
                "start_time": seconds_to_hhmmss(ws),
                "theme": theme,
                "hits": int(hits),
            })

speech_30s_windows_all = pd.DataFrame(window_rows)
speech_30s_windows = speech_30s_windows_all[speech_30s_windows_all["n_raw_words"] > 0].copy().reset_index(drop=True)
speech_30s_theme_long = pd.DataFrame(theme_long_rows)

print("N janelas totais:", len(speech_30s_windows_all))
print("N janelas com speech:", len(speech_30s_windows))

display_cols = [
    "window_id", "start_time", "end_time", "start_frame", "end_frame",
    "n_raw_words", "dominant_theme", "dominant_theme_score", "themes_present",
    "source_segments", "text"
]

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 250, "display.width", 2000):
    display(speech_30s_windows[display_cols])

# Guardar CSV sem a coluna de embedding/listas
speech_30s_windows.drop(columns=["tokens", "approx_embedding"], errors="ignore").to_csv(
    OUTPUT_DIR / "speech_30s_windows_topic_timeline.csv",
    index=False
)

speech_30s_windows_all.drop(columns=["tokens", "approx_embedding"], errors="ignore").to_csv(
    OUTPUT_DIR / "speech_30s_windows_all.csv",
    index=False
)

speech_30s_theme_long.to_csv(OUTPUT_DIR / "speech_30s_theme_timeline_long.csv", index=False)



## 5. Timeline temática por janelas de 30s

Visualização dos temas mais frequentes nas janelas speech.


In [ ]:

if not speech_30s_theme_long.empty:
    top_themes = (
        speech_30s_theme_long.groupby("theme")["hits"]
        .sum()
        .sort_values(ascending=False)
        .head(8)
        .index
        .tolist()
    )

    theme_pivot = (
        speech_30s_theme_long[speech_30s_theme_long["theme"].isin(top_themes)]
        .pivot_table(index="start_sec", columns="theme", values="hits", aggfunc="sum", fill_value=0)
        .sort_index()
    )

    plt.figure(figsize=(14, 5))
    for theme in theme_pivot.columns:
        plt.plot(theme_pivot.index / 60, theme_pivot[theme], marker="o", label=theme)

    plt.title("Menções temáticas no speech por janelas de 30s")
    plt.xlabel("Tempo do vídeo (min)")
    plt.ylabel("Nº de hits no dicionário")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("Não foram encontrados temas nas janelas speech.")



## 6. Mudança lexical e semântica entre janelas de 30s

Aqui comparamos cada janela de speech com a anterior.

- `lexical_change`: diferença nas palavras/tokens da janela;
- `semantic_change`: diferença aproximada entre embeddings, quando possível.

A mudança semântica é apenas aproximada, porque os embeddings originais são por segmento speech, não por janela de 30s.


In [ ]:

def jaccard_similarity(tokens_a, tokens_b):
    set_a = set(tokens_a) if isinstance(tokens_a, list) else set()
    set_b = set(tokens_b) if isinstance(tokens_b, list) else set()

    if len(set_a) == 0 and len(set_b) == 0:
        return np.nan
    if len(set_a | set_b) == 0:
        return np.nan
    return len(set_a & set_b) / len(set_a | set_b)


def cosine_similarity_vectors(vec_a, vec_b):
    if vec_a is None or vec_b is None:
        return np.nan
    try:
        a = np.asarray(vec_a, dtype=float)
        b = np.asarray(vec_b, dtype=float)
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        if denom == 0:
            return np.nan
        return float(np.dot(a, b) / denom)
    except Exception:
        return np.nan


speech_30s_windows = speech_30s_windows.sort_values("start_sec").reset_index(drop=True).copy()

lex_sims = [np.nan]
sem_sims = [np.nan]
time_gaps = [np.nan]

for i in range(1, len(speech_30s_windows)):
    prev = speech_30s_windows.iloc[i - 1]
    curr = speech_30s_windows.iloc[i]

    lex_sims.append(jaccard_similarity(prev["tokens"], curr["tokens"]))
    sem_sims.append(cosine_similarity_vectors(prev.get("approx_embedding"), curr.get("approx_embedding")))
    time_gaps.append(float(curr["start_sec"] - prev["end_sec"]))

speech_30s_windows["lexical_similarity_prev"] = lex_sims
speech_30s_windows["lexical_change"] = 1 - speech_30s_windows["lexical_similarity_prev"]

speech_30s_windows["semantic_similarity_prev"] = sem_sims
speech_30s_windows["semantic_change"] = 1 - speech_30s_windows["semantic_similarity_prev"]

speech_30s_windows["time_gap_from_prev_sec"] = time_gaps

# Thresholds automáticos
valid_lex = speech_30s_windows["lexical_change"].replace([np.inf, -np.inf], np.nan).dropna()
valid_sem = speech_30s_windows["semantic_change"].replace([np.inf, -np.inf], np.nan).dropna()

LEXICAL_CHANGE_THRESHOLD = float(valid_lex.quantile(LEXICAL_CHANGE_QUANTILE)) if len(valid_lex) > 0 else 0.75
SEMANTIC_CHANGE_THRESHOLD = float(valid_sem.quantile(SEMANTIC_CHANGE_QUANTILE)) if len(valid_sem) > 0 else np.inf

print("LEXICAL_CHANGE_THRESHOLD:", round(LEXICAL_CHANGE_THRESHOLD, 3))
if np.isfinite(SEMANTIC_CHANGE_THRESHOLD):
    print("SEMANTIC_CHANGE_THRESHOLD:", round(SEMANTIC_CHANGE_THRESHOLD, 3))
else:
    print("SEMANTIC_CHANGE_THRESHOLD: indisponível")

display_cols = [
    "window_id", "start_time", "end_time", "n_raw_words",
    "dominant_theme", "themes_present",
    "lexical_change", "semantic_change", "time_gap_from_prev_sec", "text"
]

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 220, "display.width", 2000):
    display(speech_30s_windows[display_cols])



## 7. Candidatos a mudança de tema/discurso por speech

Esta secção tenta encontrar onde o discurso muda entre janelas de 30s.

Isto é **divisão temática por speech**, não transição pivot → peça.


In [ ]:

def list_difference_or_empty(a, b):
    return a != b


def transition_reason_for_row(row):
    reasons = []
    if row.get("theme_changed", False):
        reasons.append("theme_changed")
    if row.get("new_theme_after_unknown", False):
        reasons.append("new_theme_after_unknown")
    if row.get("lexical_change_high", False):
        reasons.append("lexical_change_high")
    if row.get("semantic_change_high", False):
        reasons.append("semantic_change_high")
    if row.get("entities_changed", False):
        reasons.append("entities_changed")
    if row.get("volume_change_high", False):
        reasons.append("volume_change_high")
    if row.get("gap_high", False):
        reasons.append("gap_high")
    if len(reasons) == 0:
        return ""
    return ", ".join(reasons)


speech_30s_windows["prev_theme"] = speech_30s_windows["dominant_theme"].shift(1)
speech_30s_windows["prev_candidates"] = speech_30s_windows["candidates_present"].shift(1).fillna("")
speech_30s_windows["prev_parties"] = speech_30s_windows["parties_present"].shift(1).fillna("")
speech_30s_windows["prev_n_raw_words"] = speech_30s_windows["n_raw_words"].shift(1)

speech_30s_windows["theme_changed"] = (
    (speech_30s_windows["dominant_theme"] != speech_30s_windows["prev_theme"]) &
    (speech_30s_windows["dominant_theme"] != "Other/Unknown") &
    (speech_30s_windows["prev_theme"].notna()) &
    (speech_30s_windows["prev_theme"] != "Other/Unknown")
)

speech_30s_windows["new_theme_after_unknown"] = (
    (speech_30s_windows["dominant_theme"] != "Other/Unknown") &
    (speech_30s_windows["prev_theme"] == "Other/Unknown")
)

speech_30s_windows["lexical_change_high"] = speech_30s_windows["lexical_change"] >= LEXICAL_CHANGE_THRESHOLD

if np.isfinite(SEMANTIC_CHANGE_THRESHOLD):
    speech_30s_windows["semantic_change_high"] = speech_30s_windows["semantic_change"] >= SEMANTIC_CHANGE_THRESHOLD
else:
    speech_30s_windows["semantic_change_high"] = False

speech_30s_windows["entities_changed"] = (
    (speech_30s_windows["candidates_present"].fillna("") != speech_30s_windows["prev_candidates"].fillna("")) |
    (speech_30s_windows["parties_present"].fillna("") != speech_30s_windows["prev_parties"].fillna(""))
)

volume_change = (speech_30s_windows["n_raw_words"] - speech_30s_windows["prev_n_raw_words"]).abs()
valid_volume_change = volume_change.dropna()
VOLUME_CHANGE_THRESHOLD = float(valid_volume_change.quantile(VOLUME_CHANGE_QUANTILE)) if len(valid_volume_change) > 0 else np.inf
speech_30s_windows["volume_change"] = volume_change
speech_30s_windows["volume_change_high"] = speech_30s_windows["volume_change"] >= VOLUME_CHANGE_THRESHOLD

speech_30s_windows["gap_high"] = speech_30s_windows["time_gap_from_prev_sec"].fillna(0) > MAX_MERGE_GAP_SECONDS

signal_cols = [
    "theme_changed",
    "new_theme_after_unknown",
    "lexical_change_high",
    "semantic_change_high",
    "entities_changed",
    "volume_change_high",
    "gap_high",
]

speech_30s_windows["transition_score"] = speech_30s_windows[signal_cols].astype(int).sum(axis=1)
speech_30s_windows["transition_reason"] = speech_30s_windows.apply(transition_reason_for_row, axis=1)
speech_30s_windows["is_transition_candidate"] = speech_30s_windows["transition_score"] >= WINDOW_TRANSITION_SCORE_THRESHOLD

# A primeira janela com speech é sempre início do primeiro bloco.
if len(speech_30s_windows) > 0:
    speech_30s_windows.loc[0, "is_transition_candidate"] = True
    speech_30s_windows.loc[0, "transition_reason"] = "first_speech_window"
    speech_30s_windows.loc[0, "transition_score"] = max(
        int(speech_30s_windows.loc[0, "transition_score"]),
        WINDOW_TRANSITION_SCORE_THRESHOLD
    )

speech_30s_transition_candidates = speech_30s_windows[speech_30s_windows["is_transition_candidate"]].copy()

print("VOLUME_CHANGE_THRESHOLD:", round(VOLUME_CHANGE_THRESHOLD, 3) if np.isfinite(VOLUME_CHANGE_THRESHOLD) else "indisponível")
print("N candidatos a mudança temática:", len(speech_30s_transition_candidates))

display_cols = [
    "window_id", "start_time", "end_time", "start_frame", "end_frame",
    "dominant_theme", "themes_present",
    "transition_score", "transition_reason",
    "lexical_change", "semantic_change",
    "text"
]

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 250, "display.width", 2000):
    display(speech_30s_transition_candidates[display_cols])

speech_30s_transition_candidates.drop(columns=["tokens", "approx_embedding"], errors="ignore").to_csv(
    OUTPUT_DIR / "speech_30s_transition_candidates.csv",
    index=False
)



## 8. Blocos speech por tema/discurso

Agora agrupamos janelas consecutivas até aparecer uma janela candidata a transição.

Depois fazemos uma consolidação simples para evitar blocos muito curtos.


In [ ]:

def aggregate_windows_to_block(group, block_id):
    group = group.sort_values("start_sec")

    text_concat = " ".join(group["text"].dropna().astype(str).tolist())
    clean_concat = clean_text_pt(text_concat)

    theme_counts = count_alias_hits(clean_concat, THEME_ALIASES)
    candidate_counts = count_alias_hits(clean_concat, CANDIDATE_ALIASES)
    party_counts = count_alias_hits(clean_concat, PARTY_ALIASES)

    theme_hits_total = sum(theme_counts.values())

    if theme_hits_total > 0:
        dominant_theme = max(theme_counts, key=theme_counts.get)
        dominant_theme_score = int(theme_counts[dominant_theme])
    else:
        dominant_theme = "Other/Unknown"
        dominant_theme_score = 0

    source_segments = concat_unique_non_empty(group["source_segments"], max_items=50)

    return {
        "block_id": int(block_id),
        "start_sec": int(group["start_sec"].min()),
        "end_sec": int(group["end_sec"].max()),
        "start_time": seconds_to_hhmmss(group["start_sec"].min()),
        "end_time": seconds_to_hhmmss(group["end_sec"].max()),
        "start_frame": int(group["start_frame"].min()),
        "end_frame": int(group["end_frame"].max()),
        "duration_sec": int(group["end_sec"].max() - group["start_sec"].min()),
        "duration_min": float((group["end_sec"].max() - group["start_sec"].min()) / 60),
        "n_windows": int(len(group)),
        "n_raw_words": int(group["n_raw_words"].sum()),
        "source_segments": source_segments,
        "dominant_theme": dominant_theme,
        "dominant_theme_score": dominant_theme_score,
        "theme_hits_total": int(theme_hits_total),
        "themes_present": ", ".join([k for k, v in theme_counts.items() if v > 0]),
        "candidates_present": ", ".join([k for k, v in candidate_counts.items() if v > 0]),
        "parties_present": ", ".join([k for k, v in party_counts.items() if v > 0]),
        "start_transition_reason": str(group.iloc[0].get("transition_reason", "")),
        "max_transition_score": int(group["transition_score"].max()) if "transition_score" in group.columns else 0,
        "example_text": text_concat[:500],
        "text_concat": text_concat,
        "raw_window_ids": ", ".join(group["window_id"].astype(str).tolist()),
    }


speech_windows_for_blocks = speech_30s_windows.sort_values("start_sec").copy()
speech_windows_for_blocks["raw_block_id"] = speech_windows_for_blocks["is_transition_candidate"].astype(int).cumsum()

raw_blocks = []
for block_id, group in speech_windows_for_blocks.groupby("raw_block_id"):
    raw_blocks.append(aggregate_windows_to_block(group, block_id))

speech_30s_blocks_raw = pd.DataFrame(raw_blocks)

print("N blocos speech raw:", len(speech_30s_blocks_raw))

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 250, "display.width", 2000):
    display(speech_30s_blocks_raw[[
        "block_id", "start_time", "end_time", "start_frame", "end_frame",
        "duration_min", "n_windows", "n_raw_words",
        "dominant_theme", "themes_present", "start_transition_reason", "example_text"
    ]])


def block_theme_set(row):
    return split_items(row.get("themes_present", ""))


def blocks_are_similar_for_merge(prev_row, curr_row):
    prev_theme = prev_row.get("dominant_theme", "Other/Unknown")
    curr_theme = curr_row.get("dominant_theme", "Other/Unknown")

    if prev_theme == curr_theme and prev_theme != "Other/Unknown":
        return True

    prev_themes = block_theme_set(prev_row)
    curr_themes = block_theme_set(curr_row)

    if len(prev_themes & curr_themes) > 0:
        return True

    if prev_theme != "Other/Unknown" and curr_theme != "Other/Unknown":
        if theme_group(prev_theme) == theme_group(curr_theme):
            return True

    return False


def merge_block_dicts(prev_row, curr_row):
    prev = dict(prev_row)
    curr = dict(curr_row)

    text_concat = (str(prev.get("text_concat", "")) + " " + str(curr.get("text_concat", ""))).strip()
    clean_concat = clean_text_pt(text_concat)

    theme_counts = count_alias_hits(clean_concat, THEME_ALIASES)
    candidate_counts = count_alias_hits(clean_concat, CANDIDATE_ALIASES)
    party_counts = count_alias_hits(clean_concat, PARTY_ALIASES)

    theme_hits_total = sum(theme_counts.values())
    if theme_hits_total > 0:
        dominant_theme = max(theme_counts, key=theme_counts.get)
        dominant_theme_score = int(theme_counts[dominant_theme])
    else:
        dominant_theme = "Other/Unknown"
        dominant_theme_score = 0

    merged = {
        "block_id": prev.get("block_id"),
        "start_sec": int(min(prev["start_sec"], curr["start_sec"])),
        "end_sec": int(max(prev["end_sec"], curr["end_sec"])),
        "start_time": seconds_to_hhmmss(min(prev["start_sec"], curr["start_sec"])),
        "end_time": seconds_to_hhmmss(max(prev["end_sec"], curr["end_sec"])),
        "start_frame": int(min(prev["start_frame"], curr["start_frame"])),
        "end_frame": int(max(prev["end_frame"], curr["end_frame"])),
        "duration_sec": int(max(prev["end_sec"], curr["end_sec"]) - min(prev["start_sec"], curr["start_sec"])),
        "duration_min": float((max(prev["end_sec"], curr["end_sec"]) - min(prev["start_sec"], curr["start_sec"])) / 60),
        "n_windows": int(prev.get("n_windows", 0) + curr.get("n_windows", 0)),
        "n_raw_words": int(prev.get("n_raw_words", 0) + curr.get("n_raw_words", 0)),
        "source_segments": concat_unique_non_empty([prev.get("source_segments", ""), curr.get("source_segments", "")], max_items=80),
        "dominant_theme": dominant_theme,
        "dominant_theme_score": dominant_theme_score,
        "theme_hits_total": int(theme_hits_total),
        "themes_present": ", ".join([k for k, v in theme_counts.items() if v > 0]),
        "candidates_present": ", ".join([k for k, v in candidate_counts.items() if v > 0]),
        "parties_present": ", ".join([k for k, v in party_counts.items() if v > 0]),
        "start_transition_reason": prev.get("start_transition_reason", ""),
        "max_transition_score": int(max(prev.get("max_transition_score", 0), curr.get("max_transition_score", 0))),
        "example_text": text_concat[:500],
        "text_concat": text_concat,
        "raw_window_ids": concat_unique_non_empty([prev.get("raw_window_ids", ""), curr.get("raw_window_ids", "")], max_items=200),
        "raw_block_ids": concat_unique_non_empty([prev.get("raw_block_ids", str(prev.get("block_id", ""))), curr.get("raw_block_ids", str(curr.get("block_id", "")))], max_items=80),
    }

    return merged


# ------------------------------------------------------------
# Consolidação de blocos curtos
# ------------------------------------------------------------
consolidated = []

for _, row in speech_30s_blocks_raw.iterrows():
    row_dict = row.to_dict()
    row_dict["raw_block_ids"] = str(row_dict["block_id"])

    if len(consolidated) == 0:
        consolidated.append(row_dict)
        continue

    prev = consolidated[-1]
    gap = float(row_dict["start_sec"] - prev["end_sec"])

    prev_short = prev["duration_sec"] < MIN_FINAL_BLOCK_SECONDS
    curr_short = row_dict["duration_sec"] < MIN_FINAL_BLOCK_SECONDS

    similar = blocks_are_similar_for_merge(prev, row_dict)
    has_unknown = (
        prev.get("dominant_theme") == "Other/Unknown" or
        row_dict.get("dominant_theme") == "Other/Unknown"
    )

    should_merge = (
        gap <= MAX_MERGE_GAP_SECONDS and
        (prev_short or curr_short) and
        (similar or has_unknown)
    )

    if should_merge:
        consolidated[-1] = merge_block_dicts(prev, row_dict)
    else:
        consolidated.append(row_dict)

speech_30s_topic_blocks = pd.DataFrame(consolidated)

# Reatribuir IDs finais
if not speech_30s_topic_blocks.empty:
    speech_30s_topic_blocks = speech_30s_topic_blocks.sort_values("start_sec").reset_index(drop=True)
    speech_30s_topic_blocks["block_id"] = np.arange(1, len(speech_30s_topic_blocks) + 1)

print("N blocos speech consolidados:", len(speech_30s_topic_blocks))

display_cols = [
    "block_id", "start_time", "end_time", "start_frame", "end_frame",
    "duration_min", "n_windows", "n_raw_words",
    "dominant_theme", "themes_present", "candidates_present", "parties_present",
    "start_transition_reason", "raw_block_ids", "example_text"
]

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 300, "display.width", 2000):
    display(speech_30s_topic_blocks[display_cols])

speech_30s_blocks_raw.to_csv(OUTPUT_DIR / "speech_30s_blocks_raw.csv", index=False)
speech_30s_topic_blocks.to_csv(OUTPUT_DIR / "speech_30s_topic_blocks_consolidated.csv", index=False)



## 9. Tabela final e síntese

Esta é a tabela principal para usar se quiseres mostrar apenas a divisão temática speech-only.


In [ ]:

speech_topic_validation_table = speech_30s_topic_blocks.copy()

final_cols = [
    "block_id",
    "start_time",
    "end_time",
    "start_frame",
    "end_frame",
    "duration_min",
    "n_windows",
    "n_raw_words",
    "dominant_theme",
    "themes_present",
    "candidates_present",
    "parties_present",
    "start_transition_reason",
    "example_text",
]

final_cols = [c for c in final_cols if c in speech_topic_validation_table.columns]

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 350, "display.width", 2000):
    display(speech_topic_validation_table[final_cols])

speech_topic_validation_table[final_cols].to_csv(
    OUTPUT_DIR / "speech_30s_topic_validation_table.csv",
    index=False
)

summary = f"""
### Síntese speech-only por janelas de 30s

- Ficheiro analisado: `{speech_path.name}`
- Segmentos originais speech: **{len(speech)}**
- Janelas de 30s com speech: **{len(speech_30s_windows)}**
- Candidatos a mudança temática: **{len(speech_30s_transition_candidates)}**
- Blocos finais speech consolidados: **{len(speech_30s_topic_blocks)}**

Esta abordagem usa o speech para dividir o discurso em blocos temáticos aproximados.
A granularidade é melhor do que usar os segmentos originais do pickle, mas continua limitada porque não existem timestamps por palavra.
"""

display(Markdown(summary))

print("Outputs guardados em:", OUTPUT_DIR.resolve())
print("- speech_30s_windows_topic_timeline.csv")
print("- speech_30s_theme_timeline_long.csv")
print("- speech_30s_transition_candidates.csv")
print("- speech_30s_blocks_raw.csv")
print("- speech_30s_topic_blocks_consolidated.csv")
print("- speech_30s_topic_validation_table.csv")



## 10. Outputs gerados

Ficheiros principais em `outputs_speech_30s_topic_analysis/`:

- `speech_30s_windows_topic_timeline.csv` — tema por janela de 30s com speech;
- `speech_30s_windows_all.csv` — todas as janelas de 30s, incluindo vazias;
- `speech_30s_theme_timeline_long.csv` — formato long para plots de temas;
- `speech_30s_transition_candidates.csv` — possíveis mudanças temáticas entre janelas;
- `speech_30s_blocks_raw.csv` — blocos antes da consolidação;
- `speech_30s_topic_blocks_consolidated.csv` — blocos finais por tema/discurso;
- `speech_30s_topic_validation_table.csv` — tabela final mais simples para validação/apresentação.
